# 02 — Baseline and Candidate Design

This review turns the data-understanding conclusions into a finite, reviewable candidate design: which baselines anchor comparison, which model families are architecturally viable, which experiments are worth running, and what resource gates must clear first. Every table below is rendered from the persisted design configuration at execution time, so the notebook stays truthful if a configuration changes.

## Purpose and execution status

### What this review decides

This is a design review, not an experiment run. It fixes the comparison contract, enumerates candidates, and records the gates that stand between the design and a measured result. Nothing here fits a model, generates or selects features, searches hyperparameters, tunes an operating threshold, or computes a performance number.

### What has not been measured

No comparative baseline, candidate, or model-family result exists. An earlier non-comparative smoke run exercised the baseline portfolio purely for workflow continuity: it confirmed that the estimators fit and that artifacts are written, and it deliberately produced no comparative evidence. No comparative OOF artifact has been generated, every result field in the design configuration is `null`, and every gate is pending. Performance numbers are therefore reported as *not run* rather than as a value, an ordering, or a preference. Any statement about which candidate is stronger would be unsupported at this stage and is deliberately absent.

### Design inputs

| Input | Role |
| --- | --- |
| `configs/experiments/baselines.yaml` | Frozen prediction contract and the universal baseline portfolio |
| `configs/experiments/feature_campaign.yaml` | Model-family audit, context baselines, and the finite experiment matrix |
| `configs/resource_budget.yaml` | Candidate sizing, resource caps, approval gates, and stop rules |
| `outputs/splits/building_folds.parquet` | The persisted building-level assignment, loaded and never recreated |

**Purpose:** Resolve repository-relative design inputs, load the persisted configuration, and confirm the design-only gates are still closed.

In [ ]:
import json
from pathlib import Path

import polars as pl
import yaml
from IPython.display import Markdown, display

from project.paths import ProjectPaths

paths = ProjectPaths.resolve(start=Path.cwd())
config_dir = paths.root / 'configs'
budget_path = config_dir / 'resource_budget.yaml'
campaign_path = config_dir / 'experiments' / 'feature_campaign.yaml'
baseline_path = config_dir / 'experiments' / 'baselines.yaml'
split_path = paths.splits_dir / 'building_folds.parquet'
split_metadata_path = paths.splits_dir / 'split_metadata.json'
design_inputs = {'resource budget': budget_path, 'feature campaign': campaign_path,
                 'baseline portfolio': baseline_path, 'building manifest': split_path,
                 'split metadata': split_metadata_path}
missing_inputs = [name for name, path in design_inputs.items() if not path.is_file()]
if missing_inputs:
    raise FileNotFoundError(f'Required design inputs are missing: {missing_inputs}')

def read_config(path: Path) -> dict:
    return yaml.safe_load(path.read_text(encoding='utf-8'))

def compact(text: object, limit: int = 96) -> str:
    collapsed = ' '.join(str(text).split())
    return collapsed if len(collapsed) <= limit else collapsed[: limit - 1] + '\u2026'

budget = read_config(budget_path)
campaign = read_config(campaign_path)
baseline_portfolio = read_config(baseline_path)
contract = campaign['prediction_contract']
if campaign['mode'] != 'design_only' or campaign['execution_authorized']:
    raise ValueError('The campaign must stay design-only until execution is authorized')
if budget['mode'] != 'design_only' or budget['execution_authorized']:
    raise ValueError('The resource budget must stay design-only until execution is authorized')
inherited = {'target': contract['target'], 'positive_label': contract['positive_label'],
             'primary_metric': contract['primary_metric'], 'folds': contract['folds']}
frozen = {'target': baseline_portfolio['target'], 'positive_label': baseline_portfolio['positive_label'],
          'primary_metric': baseline_portfolio['primary_metric'], 'folds': baseline_portfolio['folds']}
if inherited != frozen:
    raise ValueError('The campaign contract has drifted from the frozen baseline contract')
design_gates = pl.DataFrame([
    {'gate': 'campaign mode', 'value': str(campaign['mode'])},
    {'gate': 'campaign execution_authorized', 'value': str(campaign['execution_authorized'])},
    {'gate': 'budget mode', 'value': str(budget['mode'])},
    {'gate': 'budget execution_authorized', 'value': str(budget['execution_authorized'])},
    {'gate': 'host_preflight_status', 'value': str(budget['host_preflight_status'])},
    {'gate': 'user_approval_status', 'value': str(budget['user_approval_status'])},
    {'gate': 'target / positive label', 'value': f"{contract['target']} / {contract['positive_label']}"},
    {'gate': 'primary metric', 'value': str(contract['primary_metric'])},
    {'gate': 'guardrail metrics', 'value': ', '.join(contract['guardrails'])},
    {'gate': 'split manifest', 'value': str(contract['split_manifest'])},
    {'gate': 'development folds', 'value': str(contract['folds'])},
])
display(design_gates)

**Observation:** The campaign and budget both declare `design_only` with `execution_authorized` false, and both approval gates are pending. The campaign contract matches the frozen baseline contract on target, positive label, primary metric, and fold count.

**Interpretation:** The design is internally consistent and no execution path is open. A bounded non-comparative smoke run has already verified workflow continuity. No comparative campaign run has taken place, so the notebook can describe intent but cannot report or rank comparative outcomes.

**Action:** Treat every candidate below as designed-but-unmeasured. Do not open an execution gate here; that belongs to the scripted execution stage after preflight and explicit approval.

## Inherited decisions and deferred experiments

### Decisions carried forward unchanged

The data-understanding review fixed the parts of the problem that a candidate design must not renegotiate:

- Observed-hour point-anomaly classification keyed by `(building_id, timestamp)`; no future horizon is predicted.
- `anomaly` is the target with `1` positive; ROC-AUC is primary and ranking is preserved as a continuous score.
- `building_id` is a grouping key only, never a raw predictor.
- The persisted building assignment is authoritative; no notebook-local split is permitted.
- The `gte_*` family stays pending/blocked until provenance and prediction-time availability are demonstrated.
- No operational threshold is optimized, because false-positive and false-negative costs are unspecified.
- Labeled anomalies and unusual meter readings are preserved; nothing is silently deleted or capped.

### Deferred experiments, mapped to the finite matrix

Each deferred topic from the data-understanding hand-off is resolved into one or more concrete matrix rows, or is recorded as a justified N/A or blocked item. No topic is dropped silently.

| Deferred topic | Matrix rows | Resolution |
| --- | --- | --- |
| Cleaning | `feat_preserve_raw_vs_invalid_sentinel` | Compares native raw-value handling against a verified-invalid sentinel; no silent deletion |
| Missingness | `feat_missingness_indicators`, `feat_missingness_bitmask`, `feat_missingness_passthrough_comparison` | Indicator columns, a compact bitmask, and native-passthrough versus imputation are compared separately |
| Preprocessing | `feat_safe_calendar_numeric_transforms`, `feat_histogram_quantile`, `feat_frequency_encoding`, `feat_conditional_cross_fitted_target_encoding` | All learned transforms are fold-local; target encoding is cross-fitted to respect grouped folds |
| Outlier policy | `feat_meter_deviation_ratio` | A deviation/ratio representation expresses extreme consumption without capping or deleting rows |
| Feature families and engineering | `feat_core_vs_provided`, `feat_categorical_interactions`, `feat_group_aggregates_blocked`, `blocked_unverified_gte_family` | Provided features are compared against the core set; group aggregates and the `gte_*` family stay blocked pending point-in-time and provenance review |
| PCA candidacy | `dim_pca_vs_passthrough` | Projection is compared against passthrough for distance-based families rather than assumed helpful |
| Imbalance | `regime_natural_vs_class_weighted` | Class weighting is compared against the natural distribution with the ranking metric kept primary |
| Feature selection | `dim_fold_local_selector_vs_passthrough`, `dim_semantic_block_drop` | Selection is fitted per training fold; block drops isolate which semantic group carries signal |
| Same-family tuning | `regime_same_family_hpo_vs_default` | A search is scoped inside one family so tuning gain is separable from family choice |
| Operating threshold | none — justified N/A | Costs are unspecified, so no threshold row exists; adding one would invent a decision boundary without a cost basis |
| Identifier mining | `prohibited_opaque_building_id_digit_extraction` | Explicitly prohibited: digits of an opaque identifier carry no defensible meaning for unseen buildings |

## Persisted split verification

The assignment is read from `outputs/splits/building_folds.parquet` exactly as produced; this notebook never reshuffles, re-stratifies, or falls back to a row-wise split. One manifest row is one building, which is what makes whole-building separation checkable: a building can appear in only one partition, so no hour of a building can sit on both sides of a fold boundary.

The expected shape is an isolated vault plus four building-held-out development folds. Verification below is structural and reads only the manifest and its metadata.

**Purpose:** Load the persisted manifest, confirm one row per building, verify whole-building separation across every partition, and report the vault and development-fold composition.

In [ ]:
split_manifest = pl.read_parquet(split_path)
split_metadata = json.loads(split_metadata_path.read_text(encoding='utf-8'))
required_columns = {'building_id', 'split', 'fold', 'row_count', 'anomaly_count', 'normal_count'}
if not required_columns.issubset(split_manifest.columns):
    raise ValueError('The persisted manifest is missing assignment or support fields')
if split_manifest.get_column('building_id').n_unique() != split_manifest.height:
    raise ValueError('The persisted manifest must contain exactly one row per building')
partition_members = {
    (frame.get_column('split')[0], frame.get_column('fold')[0]):
        set(frame.get_column('building_id').to_list())
    for frame in split_manifest.partition_by(['split', 'fold'])
}
partition_keys = list(partition_members)
shared = [
    (left, right)
    for index, left in enumerate(partition_keys)
    for right in partition_keys[index + 1:]
    if partition_members[left] & partition_members[right]
]
if shared:
    raise ValueError(f'Partitions must not share a building: {shared}')
development_folds = sorted(fold for split, fold in partition_keys if split == 'dev')
vault_buildings = sum(
    len(members) for (split, _fold), members in partition_members.items() if split == 'vault'
)
if len(development_folds) != contract['folds']:
    raise ValueError('The manifest fold count must match the frozen prediction contract')
if not vault_buildings:
    raise ValueError('The manifest must isolate a vault partition')
partition_summary = (
    split_manifest.group_by(['split', 'fold'], maintain_order=True)
    .agg(pl.len().alias('buildings'), pl.col('row_count').sum().alias('rows'),
         pl.col('anomaly_count').sum().alias('anomaly_rows'),
         pl.col('normal_count').sum().alias('normal_rows'))
    .sort(['split', 'fold'], nulls_last=True)
)
if partition_summary.filter((pl.col('anomaly_rows') == 0) | (pl.col('normal_rows') == 0)).height:
    raise ValueError('Every persisted partition must retain both target classes')
separation_review = pl.DataFrame([
    {'check': 'manifest buildings', 'observed': split_manifest.height},
    {'check': 'development folds', 'observed': len(development_folds)},
    {'check': 'vault buildings', 'observed': vault_buildings},
    {'check': 'partitions sharing a building', 'observed': len(shared)},
    {'check': 'recorded seed', 'observed': int(split_metadata['seed'])},
])
display(separation_review)
display(partition_summary)

**Observation:** The manifest holds one row per building, the vault is isolated, the development folds match the contracted fold count, no two partitions share a building, and every partition retains both classes.

**Interpretation:** Whole-building separation holds, so a validation fold contains only buildings the training folds never saw. That matches the unseen-building evaluation unit and is what makes any later comparison meaningful.

**Action:** Every candidate below is evaluated against this exact manifest. A row-wise or freshly generated split would silently change the question being asked and is forbidden by the stop rules.

## Baseline portfolio

### Universal baselines

Three universal baselines anchor the comparison, defined in `configs/experiments/baselines.yaml`:

- `dummy_prior` — predicts the training prior. It calibrates what "no signal" looks like under this metric and prevalence.
- `logistic_l2` — an L2-regularised linear model. It establishes how much of the signal is linearly accessible.
- `hist_gradient_boosting` — an untuned histogram gradient-boosting model. It shows what a standard non-linear learner reaches before any tuning.

These are anchors, not competitors. Their value is that they make any later gain interpretable: a candidate that cannot separate itself from the prior baseline has not demonstrated anything, and a candidate that barely moves past the linear model has not justified its extra complexity.

### Context baselines

Beyond the universal set, four context anchors are audited in the campaign configuration. Each carries a recorded status and justification rather than being quietly omitted:

- `business_incumbent` — pending input; no incumbent detector performance has been supplied, so the anchor cannot be stated.
- `temporal_naive` — N/A on architectural grounds, not resource grounds: held-out folds contain unseen buildings with no label history, so a per-building history predictor has nothing to read at prediction time.
- `grouped_hierarchical` — pending; whether point-in-time-safe building-level aggregates can be built without leakage has not been confirmed, so it is held open rather than dismissed.
- `current_champion` — pending comparison; the frozen portfolio fixes the reference family, but no campaign-variant comparison artifact exists yet.

**Purpose:** Render the universal baseline portfolio and the audited context anchors with their recorded statuses and justifications.

In [ ]:
universal_baselines = pl.DataFrame([
    {'baseline': name,
     'model_family': str(spec['model_family']),
     'recipe_id': str(spec['recipe_id']),
     'role': 'reference anchor',
     'result_status': 'not run \u2014 pending execution'}
    for name, spec in baseline_portfolio['models'].items()
])
display(universal_baselines)
context_baselines = pl.DataFrame([
    {'context_baseline': name,
     'status': str(spec.get('status')),
     'applicable': 'not asserted' if spec.get('applicable') is None else str(spec['applicable']),
     'justification': compact(spec.get('rejection_reason') or spec.get('rationale'), 120),
     'comparison_artifact': 'none \u2014 not run'}
    for name, spec in campaign['context_baselines'].items()
])
display(context_baselines)
unjustified = [
    name for name, spec in campaign['context_baselines'].items()
    if not (spec.get('rejection_reason') or spec.get('rationale'))
]
if unjustified:
    raise ValueError(f'Every context baseline must carry a recorded justification: {unjustified}')

**Observation:** All three universal baselines are defined with a recipe identifier and no recorded outcome, and all four context anchors carry an explicit status plus a written justification.

**Interpretation:** The only anchor ruled out on structural grounds is the temporal-naive one, and its reason is a property of the evaluation design rather than a shortage of compute or data. The remaining anchors are open questions with named blockers, which is a weaker claim than N/A and is the honest one here.

**Action:** Supply incumbent detector performance if an incumbent exists, and resolve the grouped/hierarchical question through point-in-time verification before treating it as either usable or inapplicable.

## Model-family audit

Eight model families are audited in `configs/experiments/feature_campaign.yaml` under `model_family_audit`. The audit separates two questions that are easy to conflate:

- **Is the family architecturally applicable?** Can it consume this feature shape and respect the building-grouped evaluation without a structural violation?
- **Is the family currently runnable?** Are the dependency and compute prerequisites approved?

### Why `blocked_unapproved_compute` is not a false N/A

A family marked `applicable: true` with `execution_status: blocked_unapproved_compute` is a real candidate that has not been authorized to run. That is a gate, not a rejection. Recording such a family as N/A would be a false N/A: it would claim the family cannot work here, when in fact it has simply not been funded with measured resources or a confirmed dependency.

The distinction matters because the two statuses have different exits. An architectural N/A is closed by an argument — as with the temporal-naive anchor, where unseen buildings structurally lack the history the method requires. A compute block is closed by measurement and approval: run the host preflight, record the caps, obtain explicit approval. Collapsing the second into the first would quietly shrink the candidate space and disguise a resourcing decision as a technical conclusion.

**Purpose:** Render the model-family audit with applicability, run gate, compatible recipes, and the recorded decision reason, then summarise the gate distribution.

In [ ]:
family_audit = pl.DataFrame([
    {'model_family': name,
     'applicable': bool(spec['applicable']),
     'execution_status': str(spec['execution_status']),
     'compatible_recipes': ', '.join(spec['compatible_recipes'] or []),
     'comparison_artifact': 'none \u2014 not run',
     'decision_reason': compact(spec['decision_reason'], 110)}
    for name, spec in campaign['model_family_audit'].items()
])
display(family_audit)
gate_distribution = (
    family_audit.group_by(['applicable', 'execution_status'])
    .agg(pl.len().alias('families'))
    .sort(['applicable', 'execution_status'])
)
display(gate_distribution)
silent_rejections = [
    name for name, spec in campaign['model_family_audit'].items()
    if not spec['applicable'] and not spec.get('rejection_reason')
]
if silent_rejections:
    raise ValueError(f'An inapplicable family must state its reason: {silent_rejections}')
if any(spec.get('oof_artifact') is not None for spec in campaign['model_family_audit'].values()):
    raise ValueError('No model family may reference a comparison artifact before execution')

**Observation:** Every audited family is marked architecturally applicable and every one sits behind the same unapproved-compute gate. None references a comparison artifact.

**Interpretation:** The candidate space is currently limited by resourcing and dependency confirmation, not by structural incompatibility with the grouped evaluation. Nothing in this table asserts that any family is preferable — applicability is a precondition for comparison, not a result of one.

**Action:** Resolve dependency availability and the compute budget through the preflight step. Families whose dependencies remain unavailable after preflight should be re-recorded with a specific reason rather than being left indefinitely blocked.

## Finite experiment matrix

The matrix under `experiment_matrix` enumerates the design as a closed list: each row is one hypothesis with a stable identifier, the thing it varies, and an empty result. It is deliberately finite. A candidate count is a design aspiration, never a quota, and padding the list with near-duplicates to reach a number is prohibited.

### Two status axes, deliberately separate

| Field | Meaning | Changed by |
| --- | --- | --- |
| `status` | Design intent — is this hypothesis worth asking at all? (`planned`, `conditional`, `blocked`, `N/A`) | A design argument |
| `execution_status` | Run gate — is this row permitted to execute right now? | Preflight, approval, or a resolved safety review |

Keeping them apart prevents a common failure: a row that is intellectually worth running but currently ungated would otherwise be recorded as abandoned, and a row that is easy to run would otherwise look justified merely because it is cheap. A row's `execution_status` never runs ahead of its model family's gate.

### Rows that stay excluded

Prohibited and blocked rows remain in the matrix as visible, argued exclusions rather than deletions. `prohibited_opaque_building_id_digit_extraction` is closed permanently on reasoning grounds. `feat_group_aggregates_blocked` and `blocked_unverified_gte_family` are closed conditionally, pending point-in-time safety and provenance review respectively. Neither class of row is counted as available work, and neither is quietly dropped from the record.

**Purpose:** Render the finite experiment matrix from the campaign configuration and verify that no row advertises a runnable gate or carries a recorded outcome.

In [ ]:
matrix_rows = campaign['experiment_matrix']
experiment_matrix = pl.DataFrame([
    {'id': str(row['id']),
     'status': str(row['status']),
     'execution_status': str(row['execution_status']),
     'model_family': 'not applicable' if row.get('model_family') is None else str(row['model_family']),
     'varies': str(row.get('varies')),
     'result': 'null \u2014 not run'}
    for row in matrix_rows
])
closed_gates = {'blocked', 'blocked_unapproved_compute', 'prohibited'}
runnable = experiment_matrix.filter(~pl.col('execution_status').is_in(list(closed_gates)))
if runnable.height:
    raise ValueError('No row may advertise a runnable gate while execution is unauthorized')
recorded_outcomes = [
    row['id'] for row in matrix_rows
    if row.get('result') is not None or row.get('oof_artifact') is not None
]
if recorded_outcomes:
    raise ValueError(f'Design-only rows must not carry outcomes: {recorded_outcomes}')
duplicate_ids = experiment_matrix.height - experiment_matrix.get_column('id').n_unique()
if duplicate_ids:
    raise ValueError('Experiment identifiers must be unique; padding is forbidden')
display(
    experiment_matrix.group_by(['status', 'execution_status'])
    .agg(pl.len().alias('rows'))
    .sort(['status', 'execution_status'])
)
display(experiment_matrix.select('id', 'status', 'execution_status', 'model_family', 'varies'))

**Observation:** Every row carries a unique identifier, a null result, and a closed run gate; the excluded rows are present with their status recorded rather than removed.

**Interpretation:** The design intent is fully enumerated while the run gate stays uniformly shut. Because nothing has executed, the matrix expresses a set of questions and carries no information about their answers.

**Action:** Keep the matrix closed and argued. Add a row only when it asks a genuinely distinct question, and reopen a blocked row only by resolving its named blocker, never by relaxing the gate.

## Resource budget and stop rules

`configs/resource_budget.yaml` separates what is known now from what can only be known on a specific machine. Candidate sizing is a design decision and is set. Every machine-dependent cap — memory, wall time, model fits, search trials, seed count, cardinality — stays `null` until a host preflight measures the actual environment and the user explicitly approves the resulting budget.

Guessing these caps before measurement would be worse than leaving them empty. A fabricated ceiling either blocks work that the host could easily absorb or authorises a run that exhausts it, and in both cases the number carries an authority it never earned. A `null` cap is an honest statement that the measurement has not happened.

### Stop rules

The stop rules are declarative constraints that must be checked programmatically before any fit, search trial, or campaign launch. In plain terms:

- No execution of any kind while caps or approval remain pending.
- The building-held-out development folds and the vault are preserved exactly as persisted.
- Row-wise or random-split fallback is forbidden outright.
- The candidate target is never padded to reach a number.
- No machine-dependent cap is set or enforced before preflight measurement is reviewed.

**Purpose:** Render candidate sizing, the pending resource caps, and the stop rules, and verify no machine-dependent cap has been set before preflight.

In [ ]:
cap_keys = ('maximum_cardinality', 'max_ram_gb', 'max_wall_time_seconds',
            'max_model_fits', 'max_hpo_trials', 'max_seed_count')
premature_caps = [key for key in cap_keys if budget.get(key) is not None]
if premature_caps:
    raise ValueError(f'Machine-dependent caps must stay unset before preflight: {premature_caps}')
resource_caps = pl.DataFrame([
    {'cap': key,
     'value': 'null \u2014 pending host preflight and approval' if budget[key] is None else str(budget[key])}
    for key in cap_keys
])
display(resource_caps)
candidate_sizing = pl.DataFrame([
    {'setting': 'candidate_target', 'value': int(budget['candidate_target']),
     'meaning': 'design aspiration, never a quota'},
    {'setting': 'candidate_cap', 'value': int(budget['candidate_cap']),
     'meaning': 'upper bound on distinct candidates'},
    {'setting': 'designed matrix rows', 'value': experiment_matrix.height,
     'meaning': 'actual finite size; fewer is acceptable, padding is forbidden'},
])
display(candidate_sizing)
if experiment_matrix.height > int(budget['candidate_cap']):
    raise ValueError('The designed matrix exceeds the candidate cap')
stop_rules = pl.DataFrame([
    {'stop_rule': str(rule['id']), 'constraint': compact(rule['description'], 130)}
    for rule in budget['stop_rules']
])
display(stop_rules)

**Observation:** Candidate sizing is set, every machine-dependent cap is null, both approval gates are pending, and the stop rules are recorded with explicit identifiers.

**Interpretation:** The budget describes a design that is ready to be measured, not a budget that has been measured. The stop rules are the mechanism that keeps that distinction enforceable rather than aspirational.

**Action:** Run the host preflight, record the measured caps, and obtain explicit approval before any gate opens. Until then the correct budget value is null, not an estimate.

## Smoke-test requirements

A smoke test is a cheap, deliberately tiny run whose only job is to prove the machinery is wired correctly before real resources are committed. It is not an experiment and produces no reportable outcome. Its value is that a broken fold loader, a misaligned feature frame, or a mis-specified metric is far cheaper to find on a handful of rows than partway through a full campaign.

### What must hold before any full run

| Requirement | Why it must precede a full run |
| --- | --- |
| Host preflight recorded and reviewed | Resource caps cannot be enforced against unmeasured capacity |
| Explicit user approval recorded | Compute spend and run authorisation are the user's decision, not an inferred one |
| Campaign execution authorized | The configuration itself must stop declaring design-only mode |
| Persisted manifest loaded, never regenerated | A regenerated split silently changes the question and breaks comparability |
| Whole-building separation re-asserted at run time | The guarantee must hold in the execution path, not only in this review |
| Learned steps fitted inside training folds only | Any transform fitted across folds contaminates the validation estimate |
| Primary and guardrail metrics computed from the frozen contract | Metric drift between design and execution makes results incomparable |
| Every run writes its own comparison artifact | A number without a reproducible artifact cannot be audited or rechecked |

### Scope of the smoke run

The smoke run should exercise one fold, one baseline, and a truncated row sample — enough to touch every stage of the path from manifest to recorded artifact. Its outputs are diagnostic only and must never be presented as performance evidence or used to prefer one candidate over another.

**Purpose:** Render the execution preconditions with their current gate values and summarise the design state as a single reader-facing statement.

In [ ]:
execution_preconditions = pl.DataFrame([
    {'requirement': 'host preflight recorded and reviewed',
     'gate_field': 'host_preflight_status', 'current_value': str(budget['host_preflight_status'])},
    {'requirement': 'explicit user approval recorded',
     'gate_field': 'user_approval_status', 'current_value': str(budget['user_approval_status'])},
    {'requirement': 'campaign execution authorized',
     'gate_field': 'campaign.execution_authorized', 'current_value': str(campaign['execution_authorized'])},
    {'requirement': 'resource caps measured, not guessed',
     'gate_field': 'resource caps', 'current_value': f'{len(cap_keys)} caps still null'},
    {'requirement': 'persisted manifest loaded, never regenerated',
     'gate_field': 'prediction_contract.split_manifest', 'current_value': str(contract['split_manifest'])},
    {'requirement': 'building-held-out folds preserved',
     'gate_field': 'prediction_contract.folds', 'current_value': str(contract['folds'])},
    {'requirement': 'metrics taken from the frozen contract',
     'gate_field': 'prediction_contract.primary_metric', 'current_value': str(contract['primary_metric'])},
])
display(execution_preconditions)
def pending_gate_names(gate_statuses: dict[str, str]) -> list[str]:
    return [
        name for name, status in gate_statuses.items()
        if str(status).casefold() != 'approved'
    ]
approval_statuses = {
    'host_preflight_status': budget['host_preflight_status'],
    'user_approval_status': budget['user_approval_status'],
}
open_gates = pending_gate_names(approval_statuses)
design_summary = (
    f'**Observation:** The design enumerates {experiment_matrix.height} finite experiment rows across '
    f'{family_audit.height} audited model families, anchored by {universal_baselines.height} universal '
    f'baselines and {context_baselines.height} context anchors. Every result field is null and '
    f'{len(cap_keys)} machine-dependent caps remain unset; open approval gates: {len(open_gates)}.\n\n'
    '**Interpretation:** The candidate space is fully specified but remains unmeasured comparatively. '
    'No comparative campaign run has taken place; the earlier smoke verified workflow continuity only. '
    'The design supports no statement about relative merit, ordering, or preference among '
    'baselines, families, or experiment rows.\n\n'
    '**Action:** Complete the host preflight, record measured caps, obtain explicit approval, then pass the '
    'smoke checks before any full run. Until a comparative OOF artifact exists, comparative outcomes stay reported as not run.'
)
display(Markdown(design_summary))